# 📊 TechCorp Dataset — Data Cleaning & Analysis

## 🎓 Welcome — Read This Before You Run Anything

This notebook is your guided walkthrough of a real-world data cleaning and analysis project using **Pandas**.
You are working with data from a fictional tech company called **TechCorp**, which has:

- **120 employees** across 6 departments
- **6 departments** with their own reference sheet
- **30 products** across 5 categories
- **400 sales transactions** from 2023 and 2024

The data was intentionally built with real-world problems — missing values, duplicate rows, wrong data types, inconsistent text, and calculation errors.
Your job, phase by phase, is to find every problem and fix it properly.

> 💡 **How to use this notebook:** Run every cell in order from top to bottom. Read the markdown explanations before running the code. Don't skip ahead.

---

## Phase 1 — Load & Explore

Before writing a single line of cleaning code, a good data analyst always asks:
*"What is this data? How many rows do I expect? What columns exist? What types are they?"*

This entire phase is about **investigation, not modification**. We are not changing anything yet.
We are building a complete picture of the dataset so we know exactly what needs to be fixed in the phases ahead.

### Step 1 — Import the Libraries

We need two libraries for this project:
- **`pandas`** — the main data analysis library. Everything we do with DataFrames lives here.
- **`numpy`** — used for numerical operations and for working with `NaN` (missing values).

You will import these at the top of every data analysis project you ever work on.

In [1]:
import pandas as pd
import numpy as np

### Step 2 — Load the Excel File

The dataset lives in a single Excel file with **4 sheets**. Instead of loading them one by one,
we pass `sheet_name=None` to `pd.read_excel()` — this tells Pandas to load **all sheets at once**
and return them as a Python **dictionary**, where each key is a sheet name and each value is a DataFrame.

This is the professional way to handle multi-sheet Excel files.

In [2]:
#Load all 4 sheets in one shot
sheets = pd.read_excel('project_data.xlsx', sheet_name=None)

### Step 3 — Confirm Which Sheets Were Loaded

Let's verify that all 4 sheets were picked up correctly.
The `.keys()` method shows us the sheet names — these become our variable names in the next step.

In [3]:
#show sheets
sheets.keys()

dict_keys(['Employees', 'Departments', 'Products', 'Sales'])

### Step 4 — Unpack Each Sheet into Its Own DataFrame

Right now all 4 sheets are sitting inside one dictionary called `sheets`. 
Let's pull each one out and give it a clean, descriptive variable name.
From this point on, we work with `employees`, `dept`, `prod`, and `sales` directly.

In [4]:
#Unpack all sheets into separate dataframes
employees = sheets['Employees']
dept = sheets['Departments']
prod = sheets['Products']
sales = sheets['Sales']

### Step 5 — First Look at the Data

`.head()` shows the first 5 rows of a DataFrame. This is always the first thing you do after loading data —
just to confirm it looks sensible and to get a feel for the columns and their content.

Look carefully at the output. Does anything look odd? Notice the `NaN` in the `salary` column — that's our first clue of a problem.

In [5]:
employees.head()

,employee_id,full_name,email,department,job_title,city,country,hire_date,salary,performance_rating,manager_id
0,1001,Gregory White,gregory.white@techcorp.com,Sales,Account Manager,Mumbai,India,2021-02-06,119704.35,4.9,1006
1,1002,Christine Sanders,christine.sanders@techcorp.com,HR,HR Generalist,Amsterdam,Netherlands,2022-06-10,54396.61,3.6,1003
2,1003,Donald Wright,donald.wright@techcorp.com,HR,Recruiter,Cape Town,South Africa,2018-11-27,55221.15,1.9,1002
3,1004,Richard Edwards,richard.edwards@techcorp.com,Engineering,QA Engineer,São Paulo,Brazil,2018-03-04,119468.82,2.6,1007
4,1005,Christopher Nelson,christopher.nelson@techcorp.com,Finance,Controller,Dubai,UAE,2022-06-29,NaN,3.2,1008


Now let's peek at the **Departments** sheet — this is our reference table.
It's small (only 6 rows) and will be used later when we merge data across sheets.

In [6]:
dept.head()

,department_id,department_name,department_head,office_location
0,101,Engineering,Alexandra Stone,San Francisco
1,102,Sales,Marcus Reid,New York
2,103,HR,Priya Sharma,London
3,104,Finance,Daniel Osei,Chicago
4,105,Marketing,Laura Chen,Berlin


### Step 6 — Check the Shape of Every Sheet

`.shape` tells you `(rows, columns)` for each DataFrame. This is **always** the first sanity check.

More importantly — compare the row counts against what you were told to expect:
- `employees` → expected **120** rows
- `sales` → expected **400** rows

If the numbers don't match, that's your first red flag. Don't ignore it — write it down.

In [ ]:
print(employees.shape)
print(dept.shape)
print(prod.shape)
print(sales.shape)

(123, 11)
(6, 4)
(35, 5)
(410, 9)


### Step 7 — Inspect the Column Names

Before you can work with a DataFrame, you need to know exactly what columns exist.
Go through each column name and ask yourself:
- What does this column represent?
- Could it have missing values?
- Does it look like it's the right data type?

This mental mapping takes 2 minutes and saves you hours of confusion later.

In [8]:
employees.columns.to_list()

['employee_id',
 'full_name',
 'email',
 'department',
 'job_title',
 'city',
 'country',
 'hire_date',
 'salary',
 'performance_rating',
 'manager_id']

In [9]:
dept.columns.to_list()

['department_id', 'department_name', 'department_head', 'office_location']

In [10]:
prod.columns.to_list()

['product_id', 'product_name', 'category', 'price', 'launch_date']

In [11]:
sales.columns.to_list()

['sale_id',
 'date',
 'employee_id',
 'product_id',
 'region',
 'units_sold',
 'unit_price',
 'discount',
 'total_revenue']

### Step 8 — Check Data Types with `.dtypes`

This is one of the most important checks you will ever run. Pandas assigns a **data type** to every column.
If the type is wrong, every calculation on that column will either crash or give you silently wrong answers.

Pay special attention to any column with **`object`** dtype that you know should be a number or a date.
For example — a `hire_date` column stored as `object` means Pandas thinks it's plain text, not a real date.
You cannot do date arithmetic on text. We will fix this in Phase 5.5.

> 🔍 **Analyst Rule:** Any column representing a date must eventually be `datetime64`, not `object`.

In [12]:
print(sales.dtypes)

sale_id            int64
date              object
employee_id        int64
product_id         int64
region            object
units_sold         int64
unit_price       float64
discount         float64
total_revenue    float64
dtype: object


### Step 9 — Full Picture with `.info()`

`.info()` is better than `.dtypes` alone because it shows you **both** data types **and** non-null counts in one view.
The non-null count is where missing values reveal themselves — if a column has 118 non-null out of 123 rows,
that means 5 values are missing.

Run `.info()` on all 4 sheets and note every column where the non-null count is less than the total row count.
Those are your missing value targets for Phase 3.

*(The cells below are commented out — uncomment the one you want to inspect.)*

In [13]:
# employees.info()
# dept.info()
# prod.info()
# sales.info()

### Step 10 — Statistical Summary with `.describe()`

`.describe()` gives you `count`, `mean`, `std`, `min`, `max`, and quartiles for every numeric column.
Use it as a **sanity check** — do the min and max values make sense?

For example:
- `salary` min should be around 40,000 and max around 150,000 — does it match?
- `performance_rating` should be between 1 and 5 — is the max 5 or is it 99?
- `discount` should be between 0 and 0.30 — any impossible values?

A wrong min or max instantly tells you there's bad data in that column.

*(Uncomment the DataFrame you want to inspect.)*

In [14]:
# employees.describe()
# dept.describe()
# prod.describe()
# sales.describe()

### Step 11 — Unique ID Check

This is a clever trick to detect duplicate rows **without even running `.duplicated()` yet**.

If `employees` has 123 rows but only 120 unique `employee_id` values — that mathematically means
3 rows share an ID that already exists. Those are your duplicates, and you've found them with a single line.

Do this check on every sheet that has a unique identifier column.

In [15]:
print(len(employees['employee_id'].unique()))

120


Let's run `.info()` specifically on Employees to get the full picture — column names,
non-null counts, and data types all at once. This confirms everything `.dtypes` told us but with the missing value counts added.

In [16]:
employees.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   employee_id         123 non-null    int64  
 1   full_name           123 non-null    object 
 2   email               123 non-null    object 
 3   department          123 non-null    object 
 4   job_title           123 non-null    object 
 5   city                123 non-null    object 
 6   country             123 non-null    object 
 7   hire_date           123 non-null    object 
 8   salary              118 non-null    float64
 9   performance_rating  118 non-null    float64
 10  manager_id          123 non-null    int64  
dtypes: float64(2), int64(2), object(7)
memory usage: 10.7+ KB


### Step 12 — Value Counts on Categorical Columns

For columns that should have a fixed set of values — like `department` or `region` —
always check what unique values actually exist using `.value_counts()`.

This catches two problems at once:
1. **Typos** — `'Enginering'` instead of `'Engineering'`
2. **Case inconsistencies** — `'sales'` and `'Sales'` appearing as two separate groups

If either of these exist, your GroupBy results will be wrong — silently, without any error.

In [17]:
employees['department'].value_counts()

department
Engineering         26
Sales               25
Customer Support    19
Finance             19
HR                  17
Marketing           17
Name: count, dtype: int64

In [18]:
sales['region'].value_counts()

region
Asia             94
Africa           88
Europe           82
North America    74
Middle East      72
Name: count, dtype: int64

In [19]:
prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    35 non-null     int64  
 1   product_name  35 non-null     object 
 2   category      35 non-null     object 
 3   price         32 non-null     float64
 4   launch_date   35 non-null     object 
dtypes: float64(1), int64(1), object(3)
memory usage: 1.5+ KB


In [20]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410 entries, 0 to 409
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sale_id        410 non-null    int64  
 1   date           410 non-null    object 
 2   employee_id    410 non-null    int64  
 3   product_id     410 non-null    int64  
 4   region         410 non-null    object 
 5   units_sold     410 non-null    int64  
 6   unit_price     410 non-null    float64
 7   discount       405 non-null    float64
 8   total_revenue  410 non-null    float64
dtypes: float64(3), int64(4), object(2)
memory usage: 29.0+ KB


### 📋 Phase 1 — Issue Log

This is what a professional data analyst produces at the end of Phase 1 — a complete written record of every problem found.
You haven't fixed anything yet. You've just documented everything. This becomes your cleaning checklist for Phases 3 through 8.

| Sheet | Issue | Detail |
|-------|-------|--------|
| **Employees** | Extra rows | 123 rows found, expected 120 → 3 duplicate rows |
| **Employees** | Missing values | `salary`: 5 missing · `performance_rating`: 5 missing |
| **Employees** | Wrong dtype | `hire_date` stored as `object` — needs datetime conversion |
| **Employees** | Text inconsistency | Some names in lowercase or UPPERCASE — needs standardisation |
| **Products** | Missing values | `price`: 3 missing values |
| **Sales** | Extra rows | 410 rows found, expected 400 → ~10 duplicate rows |
| **Sales** | Missing values | `discount`: 5 missing values |
| **Sales** | Calculation error | Some `total_revenue` values are mathematically incorrect |
| **Sales** | Wrong dtype | `date` stored as `object` — needs datetime conversion |
| **Departments** | — | ✅ No issues found — clean reference table |

> 💡 **The golden rule of Phase 1:** Never clean data you haven't understood. This issue log is proof you understood it.

---

## Phase 2 — Data Selection & Filtering

Now that you understand the structure of the dataset, let's learn to **navigate** it.
Before you clean anything, you need to be comfortable pulling exactly the slice of data you want.

This phase covers two completely separate operations:
- **Column selection** → *Which fields do I want to see?* — `df[['col1', 'col2']]`
- **Row filtering** → *Which records match my condition?* — `df[df['col'] == value]`

We then combine both using `.loc[]` — selecting specific rows **and** specific columns at the same time.

> 💡 Filtering does **not** change your original DataFrame. It creates a new one. Always assign the result to a new variable.

### Exercise 1 — Simple Column Selection

Use double brackets `df[['col1', 'col2']]` to select multiple columns.
The outer brackets are the Pandas selection operator. The inner brackets are a Python list.
Single brackets with multiple column names will throw a `KeyError` — always use double brackets.

In [21]:
# Exercise 1: Simple column selection
# Q: list of employees with just their names, department,job title, salary
employees[['full_name', 'department', 'job_title', 'salary']].head()

,full_name,department,job_title,salary
0,Gregory White,Sales,Account Manager,119704.35
1,Christine Sanders,HR,HR Generalist,54396.61
2,Donald Wright,HR,Recruiter,55221.15
3,Richard Edwards,Engineering,QA Engineer,119468.82
4,Christopher Nelson,Finance,Controller,NaN


### Exercise 2 — Filtering Rows with a Single Condition

Write the condition inside the DataFrame selector: `df[df['column'] == 'value']`.
This works by creating a boolean mask — a Series of `True`/`False` values —
and returning only the rows where the mask is `True`.

In [22]:
# Exercise 2: Single condition 
# Q: show all employees from london office
employees[employees['city'] == 'London'].head()

,employee_id,full_name,email,department,job_title,city,country,hire_date,salary,performance_rating,manager_id
55,1056,Joshua Brown,joshua.brown@techcorp.com,Engineering,Data Engineer,London,UK,2024-07-06,145878.77,2.1,1007
91,1092,Donald Thompson,donald.thompson@techcorp.com,Marketing,Marketing Specialist,London,UK,2018-01-11,84936.93,3.8,1025
114,1115,Carol Anderson,carol.anderson@techcorp.com,HR,Talent Acquisition Specialist,London,UK,2023-02-02,40356.98,3.9,1002
118,1119,Nicholas Parker,nicholas.parker@techcorp.com,Marketing,Digital Marketing Manager,London,UK,2023-04-29,89507.67,NaN,1025


### Exercise 3 — Multiple Conditions with AND (`&`)

To combine two conditions where **both** must be true, use `&` (not Python's `and`).

> ⚠️ **Critical rule:** Always wrap each individual condition in its own parentheses `()`.
> Without them, Python evaluates operator precedence incorrectly and you'll get a wrong result or an error.
>
> `df[(condition1) & (condition2)]`  ✅
>
> `df[condition1 & condition2]`  ❌

In [23]:
# Exercise 3: Multiple conditions
# # Q: show all employees from london office in the Marketing department

employees[(employees['city'] == 'London') & (employees['department'] == 'Marketing')].head()


,employee_id,full_name,email,department,job_title,city,country,hire_date,salary,performance_rating,manager_id
91,1092,Donald Thompson,donald.thompson@techcorp.com,Marketing,Marketing Specialist,London,UK,2018-01-11,84936.93,3.8,1025
118,1119,Nicholas Parker,nicholas.parker@techcorp.com,Marketing,Digital Marketing Manager,London,UK,2023-04-29,89507.67,NaN,1025


### Exercise 4 — Multiple Conditions with OR (`|`) and `.isin()`

Use `|` when **either** condition being true is enough to include the row.

For filtering on multiple values of the same column, `.isin(['val1', 'val2'])` is cleaner than chaining OR conditions.
It reads like plain English: *give me rows where department is in this list.*

In [24]:
# Exercise 3: Multiple conditions OR
# Q: show all employees from the customer-facing team - Sales and customer services
employees[(employees['department'] == 'Sales') | (employees['department'] == 'Customer Support')].head()

,employee_id,full_name,email,department,job_title,city,country,hire_date,salary,performance_rating,manager_id
0,1001,Gregory White,gregory.white@techcorp.com,Sales,Account Manager,Mumbai,India,2021-02-06,119704.35,4.9,1006
5,1006,Joseph Kelly,joseph.kelly@techcorp.com,Sales,Business Development Rep,Amsterdam,Netherlands,2022-02-19,NaN,3.4,1001
8,1009,Andrew Parker,andrew.parker@techcorp.com,Customer Support,Data Analyst,Mumbai,India,2019-09-27,60888.77,1.9,1012
11,1012,Helen Cook,helen.cook@techcorp.com,Customer Support,Support Lead,San Francisco,USA,2020-04-13,69376.19,1.5,1009
22,1023,Deborah Howard,deborah.howard@techcorp.com,Sales,Sales Manager,San Francisco,USA,2022-11-22,40186.04,4.7,1001


### ⚠️ Common Filtering Mistakes — Memorise These

These mistakes are made by every beginner at least once. Each one fails silently or with a confusing error.

| # | Mistake | Fix |
|---|---------|-----|
| 1 | `df['col1', 'col2']` — single brackets for multiple columns | `df[['col1', 'col2']]` — double brackets |
| 2 | No parentheses around individual conditions | Wrap each condition in `()` |
| 3 | Using Python `and` instead of `&` | Use `&` — Python's `and` doesn't work on Series |
| 4 | Using Python `or` instead of `\|` | Use `\|` — Python's `or` doesn't work on Series |
| 5 | Using `=` for comparison | Use `==` — single `=` is assignment, not comparison |
| 6 | Not storing the filtered result | Assign to a new variable — filtering never modifies the original |

---

## Phase 3 — Finding & Fixing Missing Values

In Pandas, a missing value is represented as `NaN` (Not a Number). It looks innocent but it's dangerous:
add any number to `NaN` and the result is `NaN`. Your averages, totals, and comparisons all break silently.

**Three strategies for handling missing values:**
1. **Drop** the row — `dropna()` — only when rows are truly expendable
2. **Fill with a fixed value** — `fillna(0)` — when zero or a known default is logically correct
3. **Fill with a smart estimate** — group median or overall mean — when you want to preserve the data's distribution

> 🔍 **Always look at the actual missing rows before deciding** how to handle them.
> A pattern in missing data (e.g. all missing values are from one department) might change your strategy entirely.

### Reload Note

We reload the data here to start Phase 3 from a clean state, in case any earlier exploration cells modified the DataFrames.
In a real project you would carry your DataFrames forward from Phase 1 — but reloading at the start of a new section is a safe habit.

In [25]:
import pandas as pd
import numpy as np

sheets = pd.read_excel('project_data.xlsx', sheet_name=None)
#Unpack all sheets into separate dataframes
employees = sheets['Employees']
dept = sheets['Departments']
prod = sheets['Products']
sales = sheets['Sales']

### Step 1 — Detect All Missing Values Across Every Sheet

The professional way to audit missing values is to loop over every sheet at once.
This gives you a complete map in one go rather than checking each sheet manually.

`.isna().sum()` counts missing values per column — it works because `True` counts as 1 and `False` as 0.
We then filter to only show columns where the count is greater than zero.

In [26]:
for name, df in sheets.items():
    print(f"\n {'='*40}")
    print(f"    Sheet: {name}")
    print(f"{'='*40}")
    missing = df.isna().sum()
    print(missing[missing > 0] if missing.any() else "No missing values") 


    Sheet: Employees
salary                5
performance_rating    5
dtype: int64

    Sheet: Departments
No missing values

    Sheet: Products
price    3
dtype: int64

    Sheet: Sales
discount    5
dtype: int64


### Step 2 — Check Missing Values as a Percentage

Count alone isn't always meaningful. 5 missing values out of 10,000 rows is negligible.
5 missing values out of 20 rows is 25% — a serious problem.

This helper function shows both the **count** and the **percentage** side by side,
so you can make an informed decision about which strategy to apply.

In [27]:
def missing_summary(df, name):
    total = len(df)
    missing = df.isna().sum()
    missing = missing[missing > 0]
    if len(missing) == 0:
        print(f"\n{name}: ✅ No missing values")
        return
    pct = (missing / total * 100).round(2)
    summary = pd.DataFrame({'missing_count': missing, 'missing_%': pct})
    print(f"\n{name}:")
    print(summary)

missing_summary(employees,   "Employees")
# missing_summary(prod,  "Products")
# missing_summary(sales, "Sales")


Employees:
                    missing_count  missing_%
salary                          5       4.07
performance_rating              5       4.07


### Step 3 — Inspect the Actual Missing Rows

Knowing *how many* values are missing is not enough. You need to see *which rows* are missing.
Sometimes the missing rows all belong to the same department, or the same time period — that pattern matters.

Here we filter to show only employees where `salary` is `NaN`, then look at their department and job title.

In [28]:
# Show employees with missing salary
print("── Employees with missing salary ──")
employees[employees['salary'].isna()]

── Employees with missing salary ──


,employee_id,full_name,email,department,job_title,city,country,hire_date,salary,performance_rating,manager_id
4,1005,Christopher Nelson,christopher.nelson@techcorp.com,Finance,Controller,Dubai,UAE,2022-06-29,NaN,3.2,1008
5,1006,Joseph Kelly,joseph.kelly@techcorp.com,Sales,Business Development Rep,Amsterdam,Netherlands,2022-02-19,NaN,3.4,1001
17,1018,carol rogers,carol.rogers@techcorp.com,HR,HR Manager,Dubai,UAE,2021-05-07,NaN,3.3,1003
38,1039,Ryan Hall,ryan.hall@techcorp.com,HR,HR Generalist,Paris,France,2018-04-07,NaN,2.6,1003
63,1064,Jonathan Price,jonathan.price@techcorp.com,Customer Support,Customer Success Manager,Tokyo,Japan,2022-11-02,NaN,3.7,1012


### Handling Strategy for `employees.salary`

The missing salaries are spread across different departments — no pattern, looks like random data entry gaps.
That means filling with a **statistical estimate** is safe and sensible.

| Option | Method | Our Decision |
|--------|--------|--------------|
| Drop rows | `dropna()` | ❌ We lose the employee's other data (name, dept, rating...) |
| Fill with fixed value | `fillna(overall_median)` | ⚠️ Same number for Engineering and HR — too imprecise |
| Fill with group estimate | `groupby().transform(median)` | ✅ **Best choice** — each dept gets its own median |

#### Option 1 — Dropping Rows (Demo Only)

Let's first see what happens if we just drop all rows with any missing value.
We save the result in a **new variable** `employees_cleaned` — we never modify the original yet.
This is just to understand what we'd lose.

In [29]:
employees_cleaned = employees.dropna()

In [30]:
employees_cleaned

,employee_id,full_name,email,department,job_title,city,country,hire_date,salary,performance_rating,manager_id
0,1001,Gregory White,gregory.white@techcorp.com,Sales,Account Manager,Mumbai,India,2021-02-06,119704.35,4.9,1006
1,1002,Christine Sanders,christine.sanders@techcorp.com,HR,HR Generalist,Amsterdam,Netherlands,2022-06-10,54396.61,3.6,1003
2,1003,Donald Wright,donald.wright@techcorp.com,HR,Recruiter,Cape Town,South Africa,2018-11-27,55221.15,1.9,1002
3,1004,Richard Edwards,richard.edwards@techcorp.com,Engineering,QA Engineer,São Paulo,Brazil,2018-03-04,119468.82,2.6,1007
6,1007,Betty Jackson,betty.jackson@techcorp.com,Engineering,Software Engineer,Paris,France,2023-05-01,107240.14,4.1,1004
...,...,...,...,...,...,...,...,...,...,...,...
117,1118,Gary Lopez,gary.lopez@techcorp.com,Finance,Finance Manager,Toronto,Canada,2021-09-27,75316.41,1.4,1008
119,1120,Cynthia Kelly,cynthia.kelly@techcorp.com,Marketing,Digital Marketing Manager,Singapore,Singapore,2018-04-28,94373.72,2.3,1013
120,1011,Jason Moore,jason.moore@techcorp.com,Finance,Financial Analyst,San Francisco,USA,2020-12-01,81701.43,1.8,1008
121,1026,Jacob Nguyen,jacob.nguyen@techcorp.com,Engineering,Senior Software Engineer,Dubai,UAE,2021-03-04,117328.63,2.8,1004


In [31]:
print(f"\nOriginal employees count: {len(employees)}")
print(f"Cleaned employees count: {len(employees_cleaned)}")


Original employees count: 123
Cleaned employees count: 113


#### Option 3 — Smart Fill: Check the Distribution First

Before filling, we need to understand the salary distribution.
`.describe()` shows us the median — and then we check the median **per department**,
because an Engineering salary is very different from an HR salary.

In [32]:
print(employees['salary'].describe())

count       118.000000
mean      98141.225763
std       32358.000110
min       40186.040000
25%       69177.350000
50%      102298.750000
75%      124714.742500
max      149829.990000
Name: salary, dtype: float64


In [33]:
print(employees.groupby('department')['salary'].median())

department
Customer Support     98764.275
Engineering         107899.335
Finance             105906.865
HR                   86249.100
Marketing           100777.760
Sales                88808.850
Name: salary, dtype: float64


In [34]:
print(employees['salary'].median())

102298.75


#### ✅ Apply the Fix — Fill Salary with Department Median

`.groupby('department')['salary'].transform(lambda x: x.fillna(x.median()))` does three things at once:
1. **Groups** employees by department
2. **Calculates** the median salary within each group
3. **Fills** only the NaN values with that group's median — leaving non-missing values untouched

The key is `.transform()` — unlike regular GroupBy which collapses rows into a summary,
transform keeps every row and just fills in the gaps.

In [35]:
employees['salary'] = employees.groupby('department')['salary'].transform(lambda x: x.fillna(x.median()))

In [36]:
employees.isna().sum()

employee_id           0
full_name             0
email                 0
department            0
job_title             0
city                  0
country               0
hire_date             0
salary                0
performance_rating    5
manager_id            0
dtype: int64

#### Fix Performance Rating — Fill with Overall Mean

Performance ratings (1–5) don't vary as dramatically by department as salary does.
The overall mean is a perfectly defensible fill here — it places the missing value
at the centre of the distribution rather than assuming a high or low score.

In [37]:
overall_mean = employees['performance_rating'].mean().round(2)

In [38]:
overall_mean

np.float64(3.05)

In [39]:
employees['performance_rating'] = employees['performance_rating'].fillna(overall_mean)

In [40]:
employees.isna().sum()

employee_id           0
full_name             0
email                 0
department            0
job_title             0
city                  0
country               0
hire_date             0
salary                0
performance_rating    0
manager_id            0
dtype: int64

### Fix Products — Missing Prices

Let's identify which products have no price. Then apply the same group-fill logic —
this time grouping by `category`, because a Laptop price range ($800–$2000)
is completely different from an Accessories range ($20–$300).
Using the overall median would significantly misrepresent the value of each product.

In [41]:
prod[prod['price'].isna()][['product_id', 'product_name', 'category', 'price']]

,product_id,product_name,category,price
7,2008,NexPhone Ultra,Phone,NaN
26,2027,CloudSync Pro,Software,NaN
33,2034,AutoReport,Software,NaN


In [42]:
print(prod.groupby('category')['price'].median())

category
Accessories      82.855
Laptop         1504.810
Phone           631.985
Software        137.260
Tablet          572.830
Name: price, dtype: float64


In [43]:
prod['price'] = prod.groupby('category')['price'].transform(lambda x: x.fillna(x.median()))

In [44]:
prod.isna().sum()

product_id      0
product_name    0
category        0
price           0
launch_date     0
dtype: int64

In [45]:
prod

,product_id,product_name,category,price,launch_date
0,2001,ProBook Elite,Laptop,1504.810,2021-05-08
1,2002,UltraSlim 14,Laptop,985.430,2023-01-28
2,2003,WorkStation X,Laptop,1585.910,2021-04-29
3,2004,DevPad Pro,Laptop,1387.220,2023-09-04
4,2005,NanoBook Air,Laptop,1926.860,2021-04-15
5,2006,SmartX One,Phone,426.170,2020-07-26
6,2007,ConnectPro 12,Phone,573.820,2021-01-28
7,2008,NexPhone Ultra,Phone,631.985,2021-06-29
8,2009,VoiceEdge S,Phone,690.150,2023-10-18
9,2010,PixelMax 5G,Phone,693.870,2022-11-05


### Fix Sales — Missing Discounts

For the Sales sheet, the missing column is `discount`. Here the fix is simple and logical:
if no discount was recorded, it almost certainly means **no discount was applied**.
Zero is the correct business value — this is a logic-driven fill, not a statistical one.

In [46]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410 entries, 0 to 409
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sale_id        410 non-null    int64  
 1   date           410 non-null    object 
 2   employee_id    410 non-null    int64  
 3   product_id     410 non-null    int64  
 4   region         410 non-null    object 
 5   units_sold     410 non-null    int64  
 6   unit_price     410 non-null    float64
 7   discount       405 non-null    float64
 8   total_revenue  410 non-null    float64
dtypes: float64(3), int64(4), object(2)
memory usage: 29.0+ KB


In [47]:
sales['discount'] = sales['discount'].fillna(0)

In [48]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410 entries, 0 to 409
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sale_id        410 non-null    int64  
 1   date           410 non-null    object 
 2   employee_id    410 non-null    int64  
 3   product_id     410 non-null    int64  
 4   region         410 non-null    object 
 5   units_sold     410 non-null    int64  
 6   unit_price     410 non-null    float64
 7   discount       410 non-null    float64
 8   total_revenue  410 non-null    float64
dtypes: float64(3), int64(4), object(2)
memory usage: 29.0+ KB


In [49]:
sales[sales['sale_id'] == 3328]

,sale_id,date,employee_id,product_id,region,units_sold,unit_price,discount,total_revenue
327,3328,2023-11-28,1008,2002,Europe,19,1025.77,0.0,18710.04


### ✅ Final Audit — Confirm Zero Missing Values Remain

After all fixes, always run the same audit loop we used at the start.
If any sheet still shows missing values, the fix for that column did not work correctly.
This should show all clean — if it doesn't, stop and investigate before moving on.

In [50]:
for name, df in sheets.items():
    print(f"\n {'='*40}")
    print(f"    Sheet: {name}")
    print(f"{'='*40}")
    missing = df.isna().sum()
    print(missing[missing > 0] if missing.any() else "No missing values") 


    Sheet: Employees
No missing values

    Sheet: Departments
No missing values

    Sheet: Products
No missing values

    Sheet: Sales
No missing values


### ✅ Phase 3 Complete — Missing Value Fixes Summary

| Sheet | Column | Strategy Used | Why |
|-------|--------|---------------|-----|
| Employees | `salary` | Median per **department** | Salary varies significantly by dept — group median is more accurate |
| Employees | `performance_rating` | Overall **mean** | Small scale (1–5), no extreme outliers, mean is a fair neutral estimate |
| Products | `price` | Median per **category** | Price range varies dramatically across categories |
| Sales | `discount` | Fixed value **0** | Business logic: no recorded discount = no discount applied |

> 💡 **The golden rule of missing values:** Filling is making an assumption. Always document what you chose and why.
> A statistic you calculated is always better than an arbitrary number.

---

## Phase 4 — Finding & Removing Duplicates

Back in Phase 1 we noticed the shapes didn't match expectations:
- `employees` → 123 rows (expected 120) — **3 extra rows**
- `sales` → 410 rows (expected 400) — **~10 extra rows**

Those extra rows are duplicate records. Duplicates are dangerous because they look like normal data —
but they silently inflate your counts, distort your averages, and corrupt every aggregation that depends on them.

> 🔍 **Always inspect before you delete.** Never blindly drop rows without seeing exactly what you're removing.

### Step 1 — Confirm Current Shapes

Let's re-confirm the shapes now that missing values are fixed.
The row counts still show the extra rows — we haven't touched duplicates yet.

In [51]:
print(employees.shape)
print(dept.shape)
print(prod.shape)
print(sales.shape)

(123, 11)
(6, 4)
(35, 5)
(410, 9)


### Step 2 — Unique ID Check

Here's the smart way to detect duplicates: count unique values in the ID column and compare to total rows.
If `len(df) != df['id_col'].nunique()`, the difference is exactly the number of duplicate rows.

This tells you *how many* duplicates exist without even running `.duplicated()` yet.

In [52]:
employees['employee_id'].nunique()

120

In [53]:
dept['department_id'].nunique()

6

In [54]:
prod['product_id'].nunique()

35

In [55]:
sales['sale_id'].nunique()

400

### Step 3 — Inspect the Duplicate Rows

Now let's actually **see** the duplicate rows before touching anything.

- `.duplicated()` with default `keep='first'` marks the second (and later) copies as `True`
- `.duplicated(keep=False)` marks **all copies** — both the original and the duplicate — as `True`

Use `keep=False` when you want to see the full picture. Use the default when you're ready to remove them.

In [56]:
employees[employees.duplicated(subset='employee_id')] #keep=False, keep='first', keep='last'

,employee_id,full_name,email,department,job_title,city,country,hire_date,salary,performance_rating,manager_id
120,1011,Jason Moore,jason.moore@techcorp.com,Finance,Financial Analyst,San Francisco,USA,2020-12-01,81701.43,1.8,1008
121,1026,Jacob Nguyen,jacob.nguyen@techcorp.com,Engineering,Senior Software Engineer,Dubai,UAE,2021-03-04,117328.63,2.8,1004
122,1051,Samantha King,samantha.king@techcorp.com,Sales,Business Development Rep,Cape Town,South Africa,2022-10-18,101699.76,1.4,1006


In [57]:
employees.duplicated(subset='employee_id').sum()

np.int64(3)

In [58]:
sales.duplicated(subset='sale_id').sum()

np.int64(10)

In [59]:
sales[sales.duplicated(subset='sale_id', keep=False)].sort_values('sale_id')

,sale_id,date,employee_id,product_id,region,units_sold,unit_price,discount,total_revenue
63,3064,2024-07-02,1003,2033,Africa,8,140.08,0.09,1019.78
403,3064,2024-07-02,1003,2033,Africa,8,140.08,0.09,1019.78
78,3079,2023-12-28,1079,2028,Asia,2,452.02,0.19,732.27
408,3079,2023-12-28,1079,2028,Asia,2,452.02,0.19,732.27
409,3106,2024-01-29,1111,2027,Europe,6,1336.20,0.22,6253.42
105,3106,2024-01-29,1111,2027,Europe,6,1336.20,0.22,6253.42
405,3130,2023-10-08,1037,2008,Africa,18,658.23,0.25,8886.10
129,3130,2023-10-08,1037,2008,Africa,18,658.23,0.25,8886.10
194,3195,2024-12-09,1114,2031,Asia,2,35.12,0.21,55.49
402,3195,2024-12-09,1114,2031,Asia,2,35.12,0.21,55.49


### Step 4 — Remove the Duplicates

`.drop_duplicates()` keeps the **first** occurrence of every row and removes all subsequent copies.
We use `inplace=True` here to modify the original DataFrame directly — this is one of the cases
where modifying the original is intentional and correct.

After dropping, always call `.reset_index(drop=True)` to renumber the rows cleanly from 0.
Without it, you'll have gaps in the index (e.g. 0, 1, 3, 5...) which causes confusion later.

In [60]:
employees.drop_duplicates(inplace=True)

In [61]:
sales.drop_duplicates(inplace=True)

In [62]:
sales.duplicated(subset='sale_id').sum()

np.int64(0)

In [63]:
employees.reset_index(drop=True, inplace=True)
sales.reset_index(drop=True, inplace=True)

### Step 5 — Verify Everything

After removing duplicates, run three verification checks:
1. **Shape** — do we now have the expected row counts?
2. **Unique IDs** — does `nunique()` now equal `len(df)`?
3. **Match statement** — a simple True/False confirmation that rows = unique IDs

All three should pass before moving on.

In [64]:
print(employees.shape)
print(sales.shape)
print(prod.shape)
print(dept.shape)

(120, 11)
(400, 9)
(35, 5)
(6, 4)


In [65]:
print(employees['employee_id'].nunique())
print(sales['sale_id'].nunique())
print(prod['product_id'].nunique())
print(dept['department_id'].nunique())

120
400
35
6


In [66]:
print(f"Total employees: {len(employees)}")
print(f"Unique employees: {employees['employee_id'].nunique()}")
print(f"Match?  {len(employees) == employees['employee_id'].nunique()}")

Total employees: 120
Unique employees: 120
Match?  True


---

## Phase 5 — Cleaning & Standardising Text Data

Numbers are either right or wrong. But text? `'john smith'`, `'John Smith'`, `'JOHN SMITH'` are
**three completely different strings** to Python — even though they represent the same person.

Text inconsistencies don't throw errors. They silently produce wrong results:
- A name search finds nothing because the casing doesn't match
- A GroupBy splits one group into two because of a capitalisation difference
- A merge fails to find a match because of a trailing space

This phase teaches you the `.str` accessor — your text cleaning toolkit in Pandas.

### Step 1 — Find the Problem: Inspect Name Values

Let's first search for a specific name to see exactly how it's stored.
If the name exists but the search returns nothing, that itself reveals a casing problem.

In [67]:
employees[employees['full_name'] == "Katherine Cooper"]

,employee_id,full_name,email,department,job_title,city,country,hire_date,salary,performance_rating,manager_id
80,1081,Katherine Cooper,katherine.cooper@techcorp.com,Engineering,Tech Lead,Cape Town,South Africa,2021-05-11,142493.16,1.4,1007


Now let's look at a broader sample of names to see the inconsistency clearly.
You should be able to spot lowercase names and ALL CAPS names mixed in with properly formatted ones.

In [68]:
employees['full_name'].head(30).tolist()

['Gregory White',
 'Christine Sanders',
 'Donald Wright',
 'Richard Edwards',
 'Christopher Nelson',
 'Joseph Kelly',
 'Betty Jackson',
 'Helen Hernandez',
 'Andrew Parker',
 'Sarah Morgan',
 'Jason Moore',
 'Helen Cook',
 'Jennifer Hall',
 'Alexander Wright',
 'Katherine Martin',
 'Catherine Bailey',
 'Timothy Harris',
 'carol rogers',
 'Eric Morris',
 'Kimberly Gomez',
 'Steven Robinson',
 'Sarah King',
 'Deborah Howard',
 'Stephanie Long',
 'Raymond Bennett',
 'Jacob Nguyen',
 'Ashley Mitchell',
 'Pamela Cox',
 'Raymond Rivera',
 'Michelle Richardson']

### Step 2 — Check Other Text Columns

While we're in text-cleaning mode, let's also verify the `city` column
to understand how many unique locations we have.

In [69]:
employees['city'].nunique()

19

### Step 3 — Check for Invisible Whitespace

Leading and trailing spaces are the sneakiest text problem — they look completely normal when printed
but break every comparison. `'John Smith'` and `'John Smith '` (with a trailing space) are not equal.

We also check for **double spaces** inside names — like `'John  Smith'` — which `.str.title()` alone won't fix.

In [70]:
has_space = employees[employees['full_name'] != employees['full_name'].str.strip()]

In [71]:
hasdouble_space = employees[employees['full_name'].str.contains('  ')]

In [72]:
hasdouble_space

,employee_id,full_name,email,department,job_title,city,country,hire_date,salary,performance_rating,manager_id


### Step 4 — The Solution: `.str.title()`

`.str.title()` capitalises the **first letter of every word** and lowercases everything else.
This is the correct format for proper names — `'john smith'` becomes `'John Smith'`,
`'ALICE BROWN'` becomes `'Alice Brown'`.

Let's preview what it would look like before applying it permanently.

In [73]:
employees['full_name'].str.title().head(20)

0          Gregory White
1      Christine Sanders
2          Donald Wright
3        Richard Edwards
4     Christopher Nelson
5           Joseph Kelly
6          Betty Jackson
7        Helen Hernandez
8          Andrew Parker
9           Sarah Morgan
10           Jason Moore
11            Helen Cook
12         Jennifer Hall
13      Alexander Wright
14      Katherine Martin
15      Catherine Bailey
16        Timothy Harris
17          Carol Rogers
18           Eric Morris
19        Kimberly Gomez
Name: full_name, dtype: object

In [74]:
name = "farhan khan"
modified_name = name.title()
modified_name

'Farhan Khan'

### Step 5 — Apply the Three-Step Fix

The correct order is always:
1. **Strip** leading/trailing whitespace first — so spaces don't interfere with casing
2. **Fix capitalisation** — apply title case
3. **Fix internal spaces** — remove any double spaces inside the name

Order matters here. Always strip before casing, not after.

In [75]:
# 1. Strip white spaces
# 2. Fix capitalization 
# 3. Fix internal spaces

In [76]:
# Step 1: Strip leading/trailing spaces
employees['full_name'] = employees['full_name'].str.strip()

# Step 2: Fix capitalization
employees['full_name'] = employees['full_name'].str.title()

# Step 3: Fix internal spaces
employees['full_name'] = employees['full_name'].str.replace('  ', ' ', regex=False)

### Step 6 — Validate the Email Column

Emails should all be lowercase and follow the format `firstname.lastname@techcorp.com`.
We run two checks:
1. Are any emails stored with uppercase characters?
2. Do all emails end with the correct domain `@techcorp.com`?

If either check returns rows, those emails need cleaning.

In [77]:
# firstname.lastname@techcorp.com

# Step 1: check for uppercase in email addresses
uppercase_emails = employees[employees['email'] != employees['email'].str.lower()]
# Step 2: check all emails end with @techcorp.com
techcorp_emails = employees[~employees['email'].str.endswith('@techcorp.com')]

In [78]:
techcorp_emails

,employee_id,full_name,email,department,job_title,city,country,hire_date,salary,performance_rating,manager_id


### Step 7 — Verify the Department Column

Even though `department` looked clean in Phase 1, always do a defensive check.
We verify the count of unique departments and list them — any typo or extra space would show up here.

In [79]:
employees['department'].nunique()

6

In [80]:
employees['department'].unique()

array(['Sales', 'HR', 'Engineering', 'Finance', 'Customer Support',
       'Marketing'], dtype=object)

### Step 8 — Verify the Sales Region Column

Same check for the `region` column in Sales.
There should be exactly 5 valid regions. Any variation in capitalisation or spelling
would split a region into multiple groups during aggregation.

In [81]:
sales['region'].nunique()

5

In [82]:
sales['region'].unique()

array(['Asia', 'Africa', 'Middle East', 'North America', 'Europe'],
      dtype=object)

### Step 9 — Final Check on Departments Sheet

The Departments sheet was clean in Phase 1 and we haven't touched it.
Let's confirm it's still intact — this is our reference table and it must stay trustworthy.

In [83]:
dept.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   department_id    6 non-null      int64 
 1   department_name  6 non-null      object
 2   department_head  6 non-null      object
 3   office_location  6 non-null      object
dtypes: int64(1), object(3)
memory usage: 320.0+ bytes


---

## Phase 5.5 — Converting Date Columns to Proper Datetime

This phase exists right here — between text cleaning and new column creation — for a specific reason:
Phase 6 creates columns like `tenure_years` and `hire_year` which **require date arithmetic**.
That arithmetic only works if the date columns are proper `datetime64` dtype, not `object` (text).

Right now, Pandas thinks `hire_date`, `launch_date`, and `date` are just strings.
Try to subtract today's date from a string and you'll get: `TypeError: unsupported operand type(s) for -: 'Timestamp' and 'str'`

We fix that here, so everything in Phase 6 works cleanly.

### Step 1 — Identify Which Columns Need Converting

We already know from our Phase 1 issue log. Three columns across three sheets:

In [84]:
# employees ---> hire date
# sales -----> date
# products -----> launch_date

### Step 2 — Look at What the Date Values Actually Look Like

Before converting, always peek at the raw values. We need to confirm the format
so Pandas can parse them correctly.

Our dates are in `YYYY-MM-DD` format — the cleanest, most unambiguous date format.
Pandas handles this automatically with no `format=` argument needed.

In [85]:
employees['hire_date'].head()

0    2021-02-06
1    2022-06-10
2    2018-11-27
3    2018-03-04
4    2022-06-29
Name: hire_date, dtype: object

### Step 3 — Understanding `pd.to_datetime()`

The conversion tool is `pd.to_datetime()`. Pay attention to the `errors` parameter:
- `errors='raise'` — crashes if any value can't be converted (the default)
- `errors='coerce'` — converts what it can, turns failures into `NaT` (Not a Time)
- `errors='ignore'` — leaves unconvertible values unchanged

We always use `errors='coerce'` in production — it's safer and never crashes your script.

In [86]:
import pandas as pd
today = pd.to_datetime('today')
print(type(today))
# today - employees['hire_date']

<class 'pandas._libs.tslibs.timestamps.Timestamp'>


In [87]:
# pd.to_datetime()
# pd.to_datetime(employees['hire_date'])
# pd.to_datetime(column, the column to convert
# format=None,  the date format if needed - e.g: '%Y-%m-%d'
# errors='raise' what to do if conversion fails - 'raise', 'coerce', 'ignore' )

#raise is the default value
#coerce converts what it can and turns the rest into NaT (Not a Time)
#ignore ignores the conversion errors original value unchanged

### Step 4 — Confirm the Date Formats

Let's print sample values from all three date columns side by side.
This confirms the format before we commit to the conversion.

In [88]:
print(f"Sample hire date: employee_hire_date = {employees['hire_date'].head()}")
print(f"Sample Launch date: product_launch_date = {prod['launch_date'].head()}")
print(f"Sample sale date: sale_date = {sales['date'].head()}")

Sample hire date: employee_hire_date = 0    2021-02-06
1    2022-06-10
2    2018-11-27
3    2018-03-04
4    2022-06-29
Name: hire_date, dtype: object
Sample Launch date: product_launch_date = 0    2021-05-08
1    2023-01-28
2    2021-04-29
3    2023-09-04
4    2021-04-15
Name: launch_date, dtype: object
Sample sale date: sale_date = 0    2023-03-07
1    2023-08-15
2    2023-03-04
3    2023-03-17
4    2024-10-01
Name: date, dtype: object


In [89]:
#YYYY-MM-DD
# 15/03/2019 
# March 15, 2019


### Step 5 — Safety Check: Test the Conversion First

Professional analysts never convert blindly. We test first using `errors='coerce'`
and check if any values became `NaT` — that would mean they couldn't be parsed.

A value like `'N/A'` written as text, or an impossible date like `'31/02/2023'`,
would fail silently here. Zero failures means we're clear to proceed.

In [90]:
tested_convert = pd.to_datetime(employees['hire_date'], errors='coerce')

failed = employees[tested_convert.isna() & employees['hire_date'].notna()]

print(f"hire_date values that failed conversion: {len(failed)}")

test_prod = pd.to_datetime(prod['launch_date'], errors='coerce')
test_sales = pd.to_datetime(sales['date'], errors='coerce')

failed_prod = prod[test_prod.isna() & prod['launch_date'].notna()]
failed_sales = sales[test_sales.isna() & sales['date'].notna()]

print(f"launch_date values that failed conversion: {len(failed_prod)}")
print(f"date values that failed conversion: {len(failed_sales)}")

hire_date values that failed conversion: 0
launch_date values that failed conversion: 0
date values that failed conversion: 0


### Step 6 — Convert All Three Date Columns

Now we apply the conversion permanently to all three DataFrames.
After this cell runs, `hire_date`, `launch_date`, and `date` are all proper `datetime64[ns]` columns.

In [91]:
# Convert Employees hire_date to datetime
employees['hire_date'] = pd.to_datetime(employees['hire_date'], errors='coerce')

# Convert Products launch_date to datetime
prod['launch_date'] = pd.to_datetime(prod['launch_date'], errors='coerce')

# Convert Sales date to datetime
sales['date'] = pd.to_datetime(sales['date'], errors='coerce')

### Step 7 — Verify the Conversion

Check two things:
1. **Dtypes** — all three should now show `datetime64[ns]`, not `object`
2. **NaT count** — should be zero, confirming no values were lost during conversion

In [92]:
print("++++++ Conversion Results ++++++")

print(f" Employees hire_date: {employees['hire_date'].dtype}")
print(f" Products launch_date: {prod['launch_date'].dtype}")
print(f" Sales date: {sales['date'].dtype}")    

++++++ Conversion Results ++++++
 Employees hire_date: datetime64[ns]
 Products launch_date: datetime64[ns]
 Sales date: datetime64[ns]


In [93]:
sales['date'].isna().sum()


np.int64(0)

### Step 8 — Confirm the `.dt` Accessor Works

The real proof that conversion worked is being able to use the `.dt` accessor without errors.
`.dt` is a special property that unlocks date-specific operations — it only exists on `datetime64` columns.
If `hire_date` were still `object`, this would throw an `AttributeError`.

In [94]:
# .dt accessor for datetime columns

print(" ________ date-based columns distribution ________ ")
year_result = employees['hire_date'].dt.year
month_result = employees['hire_date'].dt.month
day_result = employees['hire_date'].dt.day

print("______ Employees hire year distribution ________ ")
print(year_result.head())
print("______ Employees hire month distribution ________ ")
print(month_result.head())
print("______ Employees hire day distribution ________ ")
print(day_result.head())

 ________ date-based columns distribution ________ 
______ Employees hire year distribution ________ 
0    2021
1    2022
2    2018
3    2018
4    2022
Name: hire_date, dtype: int32
______ Employees hire month distribution ________ 
0     2
1     6
2    11
3     3
4     6
Name: hire_date, dtype: int32
______ Employees hire day distribution ________ 
0     6
1    10
2    27
3     4
4    29
Name: hire_date, dtype: int32


### Quick Reference — All `.dt` Properties Available

Now that the columns are proper datetimes, here is everything the `.dt` accessor unlocks.
You will use most of these in Phase 6 and Phase 8.
They are all commented out here as a reference — uncomment any one to try it.

In [95]:
# Quick References - Datetime properties you can use

# employees['hire_date'].dt.year
# employees['hire_date'].dt.month
# employees['hire_date'].dt.day
# employees['hire_date'].dt.week
# employees['hire_date'].dt.weekday
# employees['hire_date'].dt.day_name()
# employees['hire_date'].dt.is_leap_year()


### Date Arithmetic — The Whole Point of This Phase

This is what we converted the columns for. Subtracting one datetime from another
gives a `Timedelta` object — a duration. From that you can extract `.dt.days`, divide by 365,
and get employee tenure in years.

This is impossible on `object` dtype columns. This is why Phase 5.5 exists.

In [96]:
today = pd.to_datetime('today')
print(today)
today - employees['hire_date'].head()

2026-05-31 22:01:12.449612


0   1940 days 22:01:12.449612
1   1451 days 22:01:12.449612
2   2742 days 22:01:12.449612
3   3010 days 22:01:12.449612
4   1432 days 22:01:12.449612
Name: hire_date, dtype: timedelta64[ns]

### Phase 5.5 — Key Concepts Summary

Before moving on, make sure these concepts are clear in your mind.

In [97]:
# object dtype
# datetime64[ns]
# pd.to_datetime()
# errors = 'coerce'  ---> NaT
# NaT = The datetime equivalent of NaN - Not a time
#  .dt 

---

## Phase 6 — Sorting & Creating New Columns

Every phase so far has been about **fixing problems** — removing bad things from the dataset.
Phase 6 is the first phase where you stop fixing and start **building**.

You will:
- Sort data to answer ranking questions immediately
- Create new columns derived from existing ones (tenure, seniority, salary band, performance flag)
- Fix the 5 intentionally wrong revenue values that have been sitting in the Sales sheet since Phase 1

> 💡 **Golden rule:** Never modify an original column to store a derived value — always create a **new column** with a descriptive name.
> If your calculation turns out to be wrong, you can still go back to the original.

---

### Part A — Sorting Data with `.sort_values()`

Sorting sounds simple, but done properly it answers real business questions immediately —
without any aggregation or complex code.

**Syntax:** `df.sort_values(by='column_name', ascending=False)`
- `ascending=True` → smallest to largest, A to Z, oldest to newest
- `ascending=False` → largest to smallest, Z to A, newest to oldest
- Pass a **list** to `by` and `ascending` to sort by multiple columns

In [98]:
# df.sort_values(by='column_name', ascending=False)

#### Sort Employees by Hire Date — Most Recently Hired First

This works correctly now because `hire_date` is a proper `datetime64` column after Phase 5.5.
If it were still stored as text, this would sort alphabetically — giving completely wrong results.

In [99]:
employees.sort_values(by='hire_date' , ascending=False)[['full_name', 'department', 'job_title', 'salary', 'hire_date']].head()

,full_name,department,job_title,salary,hire_date
78,Ryan Green,Engineering,Software Engineer,149829.99,2024-12-19
99,Kenneth Torres,Sales,Business Development Rep,112017.62,2024-10-20
107,David Young,Engineering,QA Engineer,105573.95,2024-10-19
64,Nicholas Bailey,Finance,Controller,44384.69,2024-09-15
89,Eric Carter,Finance,Controller,148903.44,2024-08-07


#### Sort Products by Launch Date — Oldest First

Which products have been in the market the longest?
Sorting by `launch_date` ascending answers that immediately.

In [100]:
prod.sort_values(by='launch_date', ascending=True)[['product_name', 'category', 'price', 'launch_date']].head()

,product_name,category,price,launch_date
31,HR Portal,Software,191.96,2019-01-31
17,USB-C Hub,Accessories,34.16,2019-02-13
13,DigiPad X,Tablet,869.41,2019-09-08
19,Noise-Cancel Headset,Accessories,240.35,2019-09-15
27,DataVault,Software,494.01,2019-10-24


#### Sort Sales by Date — Earliest Transaction First

A quick look at the oldest sales in the dataset to confirm the date range is correct.

In [101]:
sales.sort_values(by='date', ascending=True)[['sale_id', 'product_id', 'employee_id', 'discount', 'date']].head()

,sale_id,product_id,employee_id,discount,date
64,3065,2023,1079,0.20,2023-01-02
295,3296,2027,1120,0.21,2023-01-07
375,3376,2017,1029,0.21,2023-01-09
296,3297,2001,1043,0.16,2023-01-10
187,3188,2034,1003,0.24,2023-01-13


### Part B — Creating New Columns

A raw dataset gives you ingredients. New columns are the recipes you cook from those ingredients.

**The pattern is always the same:**
```python
df['new_column_name'] = some_calculation_on_existing_columns
```

You create the column name on the left side and tell Pandas what to fill it with on the right.
If the column name already exists, Pandas overwrites it. If it's new, Pandas creates it.

In [102]:
# df['new_column'] = some calculations on the existing columns

#### New Column 1 — Hire Year

The simplest new column — extract the year from `hire_date` using the `.dt` accessor.
This gives us a clean integer year for grouping and filtering later.

In [103]:
# Hiring Year of employees
employees['hire_year'] = employees['hire_date'].dt.year

print(employees[['full_name', 'hire_date', 'hire_year']].head())

            full_name  hire_date  hire_year
0       Gregory White 2021-02-06       2021
1   Christine Sanders 2022-06-10       2022
2       Donald Wright 2018-11-27       2018
3     Richard Edwards 2018-03-04       2018
4  Christopher Nelson 2022-06-29       2022


#### New Column 2 — Tenure in Years

How long has each employee worked at TechCorp? We subtract their `hire_date` from today.

The subtraction gives a `Timedelta` (a duration object). We extract `.dt.days` from it,
then divide by 365 to convert days into years. `.round(2)` keeps it to 2 decimal places.

In [104]:
today = pd.to_datetime('today')
employees['tenure_years'] = ((today - employees['hire_date']).dt.days / 365).round(2)
print(employees[['full_name', 'hire_date', 'tenure_years']].head())

            full_name  hire_date  tenure_years
0       Gregory White 2021-02-06          5.32
1   Christine Sanders 2022-06-10          3.98
2       Donald Wright 2018-11-27          7.51
3     Richard Edwards 2018-03-04          8.25
4  Christopher Nelson 2022-06-29          3.92


#### New Column 3 — Seniority Label (Using `lambda`)

Here we introduce `.apply()` with a `lambda` function — one of the most useful patterns in Pandas.

A `lambda` is a tiny anonymous function you write inline. Instead of defining a full function separately,
you write the logic right inside `.apply()`. It receives one value at a time (one row's `tenure_years`)
and returns the seniority label for that row.

Think of it as: *"for each value, apply this rule and return a result."*

In [105]:
employees['Seniority'] = employees['tenure_years'].apply(
    lambda years: 'Senior' if years >= 5 else 
                   'Mid-Level' if years >= 2 else
                   'Junior'
)

In [106]:
print(employees[['full_name', 'hire_year', 'tenure_years', 'Seniority']].head())

            full_name  hire_year  tenure_years  Seniority
0       Gregory White       2021          5.32     Senior
1   Christine Sanders       2022          3.98  Mid-Level
2       Donald Wright       2018          7.51     Senior
3     Richard Edwards       2018          8.25     Senior
4  Christopher Nelson       2022          3.92  Mid-Level


#### New Column 4 — Salary Band

Same pattern — a label based on numeric thresholds.
This converts a continuous number (salary) into a category (Low / Medium / High),
which is easier to group and visualise.

In [107]:
employees['salary_band'] = employees['salary'].apply(
    lambda salary: 'High' if salary >= 100000 else
                   'Medium' if salary >= 60000 else
                   'Low'
)

In [108]:
employees[['hire_year', 'tenure_years', 'Seniority', 'salary', 'salary_band']].head()

,hire_year,tenure_years,Seniority,salary,salary_band
0,2021,5.32,Senior,119704.350,High
1,2022,3.98,Mid-Level,54396.610,Low
2,2018,7.51,Senior,55221.150,Low
3,2018,8.25,Senior,119468.820,High
4,2022,3.92,Mid-Level,105906.865,High


#### New Column 5 — Top Performer Flag

A two-condition boolean flag. An employee is a top performer if they have
**both** a high rating AND a high salary.

We use direct boolean logic here instead of `.apply()` — it's faster, cleaner, and more Pandas-idiomatic.
The result is a `True`/`False` column, which we then map to `'Yes'`/`'No'` for readability.

In [109]:
employees['top_performer'] = ((employees['performance_rating'] >=4.0) & (employees['salary'] > 80000))

In [110]:
employees['top_performer'].head()

0     True
1    False
2    False
3    False
4    False
Name: top_performer, dtype: bool

In [111]:
employees['top_performer'] = employees['top_performer'].map({True: 'Yes', False: 'No'})

#### New Column 6 — Years Since Product Launch

Applying the same tenure logic to the Products sheet.
This tells us how long each product has been on the market.

In [112]:
today = pd.to_datetime('today')
prod['year_since_launch']  =  ((today - prod['launch_date']).dt.days / 365).round(1)


In [113]:
print(prod[['product_name', 'launch_date', 'year_since_launch']].head())

    product_name launch_date  year_since_launch
0  ProBook Elite  2021-05-08                5.1
1   UltraSlim 14  2023-01-28                3.3
2  WorkStation X  2021-04-29                5.1
3     DevPad Pro  2023-09-04                2.7
4   NanoBook Air  2021-04-15                5.1


#### New Columns 7–10 — Sales Date Dimensions

We extract multiple time dimensions from the `date` column in Sales.
These new columns (`sale_year`, `sale_month`, `sale_month_name`, `sale_quarter`) will be
essential in Phase 7 when we group sales by time period to see trends.

In [114]:
sales['sale_year'] = sales['date'].dt.year
sales['sale_month'] = sales['date'].dt.month
sales['sale_month_name'] = sales['date'].dt.month_name()
sales['sale_quarter'] = sales['date'].dt.quarter


In [115]:
sales.columns.to_list()

['sale_id',
 'date',
 'employee_id',
 'product_id',
 'region',
 'units_sold',
 'unit_price',
 'discount',
 'total_revenue',
 'sale_year',
 'sale_month',
 'sale_month_name',
 'sale_quarter']

In [116]:
sales[['sale_id', 'date','sale_year', 'sale_month', 'sale_month_name', 'sale_quarter']].head()

,sale_id,date,sale_year,sale_month,sale_month_name,sale_quarter
0,3001,2023-03-07,2023,3,March,1
1,3002,2023-08-15,2023,8,August,3
2,3003,2023-03-04,2023,3,March,1
3,3004,2023-03-17,2023,3,March,1
4,3005,2024-10-01,2024,10,October,4


In [117]:
sales.columns.to_list()

['sale_id',
 'date',
 'employee_id',
 'product_id',
 'region',
 'units_sold',
 'unit_price',
 'discount',
 'total_revenue',
 'sale_year',
 'sale_month',
 'sale_month_name',
 'sale_quarter']

In [118]:
print(employees.shape)
print(dept.shape)
print(prod.shape)
print(sales.shape)

# (123, 11)
# (6, 4)
# (35, 5)
# (410, 9)

(120, 16)
(6, 4)
(35, 6)
(400, 13)


In [120]:
sales.columns.to_list()

['sale_id',
 'date',
 'employee_id',
 'product_id',
 'region',
 'units_sold',
 'unit_price',
 'discount',
 'total_revenue',
 'sale_year',
 'sale_month',
 'sale_month_name',
 'sale_quarter']

### Part C — Fixing Incorrect Revenue Values

This is the most important task in Phase 6. Back in Phase 1, we noted that the Sales sheet has
5 rows where `total_revenue` was intentionally set to an incorrect value.
Those errors have been sitting there this whole time. Now we fix them.

**The correct formula is:**
```
total_revenue = units_sold × unit_price × (1 - discount)
```

Our approach:
1. Recalculate the correct revenue from scratch
2. Find rows where `total_revenue` doesn't match our calculation
3. Measure the discrepancy
4. Overwrite the wrong values with the correct ones

In [ ]:
# total_revenue = units_sold * unit_price * (1 - discount)

#### Step 1 — Recalculate Correct Revenue

We create a new column `correct_revenue` using the formula.
We do **not** overwrite `total_revenue` yet — we keep both so we can compare them.

In [123]:
sales['correct_revenue'] = (sales['units_sold'] * sales['unit_price'] * (1 - sales['discount'])).round(2)

In [124]:
sales['correct_revenue'].head()

0     2089.78
1    11165.46
2      891.97
3     7607.04
4      678.54
Name: correct_revenue, dtype: float64

In [125]:
sales.columns.to_list()

['sale_id',
 'date',
 'employee_id',
 'product_id',
 'region',
 'units_sold',
 'unit_price',
 'discount',
 'total_revenue',
 'sale_year',
 'sale_month',
 'sale_month_name',
 'sale_quarter',
 'correct_revenue']

#### Step 2 — Find All Rows Where the Values Don't Match

Filter to rows where `correct_revenue != total_revenue`.
This should give us exactly 5 rows — the intentional errors we've been tracking since Phase 1.

In [127]:
bad_revenue = sales[sales['correct_revenue'] != sales['total_revenue']]

In [ ]:
bad_revenue

10

#### Step 3 — Measure the Total Discrepancy

Before fixing, let's quantify how wrong the values are.
In a real company, this number represents incorrectly reported revenue in financial statements.

In [131]:
bad_revenue['discrepencies'] = (bad_revenue['correct_revenue'] - bad_revenue['total_revenue']).abs().round(2)

C:\Users\hp\AppData\Local\Temp\ipykernel_22816\1214412644.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bad_revenue['discrepencies'] = (bad_revenue['correct_revenue'] - bad_revenue['total_revenue']).abs().round(2)


In [133]:
bad_revenue['discrepencies'].sum()

np.float64(19784.43)

#### Step 4 — Fix the Values and Clean Up

Now overwrite `total_revenue` with our correctly calculated values.
Then drop the helper column `correct_revenue` — it served its purpose and we don't need it anymore.

In [134]:
sales['total_revenue'] = sales['correct_revenue']

In [136]:
sales.columns.to_list()

['sale_id',
 'date',
 'employee_id',
 'product_id',
 'region',
 'units_sold',
 'unit_price',
 'discount',
 'total_revenue',
 'sale_year',
 'sale_month',
 'sale_month_name',
 'sale_quarter',
 'correct_revenue']

In [137]:
sales = sales.drop(columns=['correct_revenue'])

In [138]:
sales.columns.to_list()

['sale_id',
 'date',
 'employee_id',
 'product_id',
 'region',
 'units_sold',
 'unit_price',
 'discount',
 'total_revenue',
 'sale_year',
 'sale_month',
 'sale_month_name',
 'sale_quarter']

---

## Phase 7 — GroupBy & Aggregation

Up until now, every operation has been at the **row level** — looking at individual employees or individual sales.
Phase 7 is where you zoom out and see the **big picture**.

GroupBy follows a three-step mental model called **Split → Apply → Combine:**
1. **Split** — divide the DataFrame into groups based on a column (e.g. one group per department)
2. **Apply** — run a calculation independently on each group (e.g. mean salary)
3. **Combine** — bring all the group results back together into one clean summary table

**Basic syntax:**
```python
df.groupby('group_column')['value_column'].aggregation_function()
```

> 💡 Always chain `.reset_index()` after `.groupby().agg()`. Without it, your group columns become the index —
> which looks confusing and makes further operations harder.

### The Simplest Aggregation First

Before GroupBy, let's remember what a plain aggregation looks like — just the overall mean salary across all employees.
GroupBy does the same thing, but independently for each group.

In [140]:
employees['salary'].mean()

np.float64(97882.60841666667)

### The Split → Apply → Combine Concept

These comments represent the mental model behind every GroupBy operation.
Keep this in mind whenever you write a GroupBy — which step are you currently defining?

In [ ]:
# Split -> Apply -> Combine

In [ ]:
# df.groupby('column_to_group_by')['column_to_aggregate'].agg_function()

### Group by Department — Average Salary

One line. Six department averages. Sorted from highest to lowest.
This immediately tells your manager which departments have the highest salary cost.

Notice how `.sort_values()` chains directly onto the GroupBy result —
the output is a Series and Series supports sorting.

In [143]:
employees.groupby('department')['salary'].mean().round(2).sort_values(ascending=False)

department
Finance             106674.74
Marketing           101631.97
Engineering          99261.37
Customer Support     96917.14
HR                   93901.71
Sales                90780.64
Name: salary, dtype: float64

### Counting Rows per Group with `.size()`

To count headcount per department, use `.size()` — **not** `.count()`.

The difference matters:
- `.size()` counts **every row**, including those with `NaN`
- `.count()` counts only **non-null values** in a specific column

For headcount, you want every employee counted regardless of whether a field is missing.

In [145]:
# size()
headcount = employees.groupby('department').size()

In [146]:
headcount

department
Customer Support    19
Engineering         25
Finance             18
HR                  17
Marketing           17
Sales               24
dtype: int64

In [147]:
# size() vs count()

### Multiple Aggregations at Once with `.agg()`

Instead of running mean, then sum, then count separately — `.agg()` lets you calculate
all of them in one shot. Pass a list of aggregation function names as strings.

In [148]:
# agg()
employees.groupby('department')['salary'].agg(['mean', 'sum', 'count']).round(2)

,mean,sum,count
department,,,
Customer Support,96917.14,1841425.62,19
Engineering,99261.37,2481534.20,25
Finance,106674.74,1920145.34,18
HR,93901.71,1596329.00,17
Marketing,101631.97,1727743.56,17
Sales,90780.64,2178735.28,24


### Named Aggregations — The Professional Way

The output above has column names like `mean`, `sum`, `count` — technically correct but not readable in a report.
Named aggregations let you define meaningful column names yourself.

**Syntax:**
```python
df.groupby('col').agg(
    output_name = ('source_column', 'function'),
)
```

You can even pass a `lambda` as the function — that's how `top_performers` is counted here.

In [152]:
dept_summary = employees.groupby('department').agg(
    head_count = ('employee_id', 'size'),
    avg_salary = ('salary', 'mean'),
    median_salary = ('salary', 'median'),
    max_salary = ('salary', 'max'),
    avg_rating = ('performance_rating', 'mean'),
    top_performers = ('top_performer', lambda x: (x == 'Yes').sum())
)

In [153]:
dept_summary

,head_count,avg_salary,median_salary,max_salary,avg_rating,top_performers
department,,,,,,
Customer Support,19,96917.138158,98764.2750,149664.00,2.921053,2
Engineering,25,99261.368000,107240.1400,149829.99,3.310000,5
Finance,18,106674.741389,106294.9175,148903.44,2.913889,5
HR,17,93901.705882,86249.1000,148725.54,3.264706,4
Marketing,17,101631.974118,100777.7600,147172.44,2.991176,2
Sales,24,90780.636667,86738.5600,144066.82,3.004167,4


### GroupBy on Sales — Revenue Analysis

Now let's move to the Sales sheet and start answering real revenue questions.
First, a quick `.head()` to remind ourselves what Sales looks like at this point in the project.

In [154]:
sales.head()

,sale_id,date,employee_id,product_id,region,units_sold,unit_price,discount,total_revenue,sale_year,sale_month,sale_month_name,sale_quarter
0,3001,2023-03-07,1061,2029,Asia,17,142.94,0.14,2089.78,2023,3,March,1
1,3002,2023-08-15,1079,2003,Africa,10,1572.60,0.29,11165.46,2023,8,August,3
2,3003,2023-03-04,1062,2026,Middle East,4,305.47,0.27,891.97,2023,3,March,1
3,3004,2023-03-17,1116,2006,Asia,20,396.20,0.04,7607.04,2023,3,March,1
4,3005,2024-10-01,1082,2021,Middle East,20,39.45,0.14,678.54,2024,10,October,4


#### Revenue by Year and Region Combined

Grouping by **two columns** gives you a multi-level result — revenue broken down
by every combination of year and region.
This lets you compare the same region year-over-year.

In [157]:
sales.groupby(['sale_year','region'])['total_revenue'].sum().round(2)

sale_year  region       
2023       Africa           195998.38
           Asia             149275.18
           Europe           147793.25
           Middle East      150462.38
           North America    183927.72
2024       Africa           186223.65
           Asia             154026.25
           Europe           200748.30
           Middle East      132071.51
           North America    207890.91
Name: total_revenue, dtype: float64

#### Total Revenue by Region

Which region generates the most revenue overall? This aggregation answers that in one line.
We sort descending, reset the index, and rename the column for a clean, report-ready output.

In [158]:
revenue_by_region = (
    sales.groupby('region')['total_revenue'].sum().round(2).sort_values(ascending=False).reset_index()
)

In [159]:
revenue_by_region

,region,total_revenue
0,North America,391818.63
1,Africa,382222.03
2,Europe,348541.55
3,Asia,303301.43
4,Middle East,282533.89


#### Quarterly Revenue Breakdown

Using the `sale_year` and `sale_quarter` columns we created in Phase 6,
we can now see revenue trending quarter by quarter.
Is Q4 always the strongest? Is revenue growing year over year?
This groupby makes those questions answerable instantly.

In [160]:
sales.groupby(['sale_year', 'sale_quarter']).agg(
    total_revenue = ('total_revenue', 'sum'),
    num_transactions = ('sale_id', 'count')
).round(2).reset_index()

,sale_year,sale_quarter,total_revenue,num_transactions
0,2023,1,160629.24,45
1,2023,2,210541.86,46
2,2023,3,207062.60,43
3,2023,4,249223.21,52
4,2024,1,219696.69,53
5,2024,2,269330.50,59
6,2024,3,201716.46,49
7,2024,4,190216.97,53


### Value Counts on Categorical Columns

`.value_counts()` is GroupBy's simpler cousin — it counts how many times each unique value
appears in a column without any additional aggregation.

Use it for quick frequency checks on categorical columns like region, seniority band, or salary band.

In [166]:
# employees['Seniority'].value_counts()
# employees['salary_band'].value_counts()
# sales['region'].value_counts()
sales['sale_month_name'].value_counts()

sale_month_name
October      44
April        40
January      38
July         36
June         33
December     33
March        33
May          32
September    29
November     28
August       27
February     27
Name: count, dtype: int64

In [169]:
employees['Seniority'].value_counts()

Seniority
Senior       62
Mid-Level    48
Junior       10
Name: count, dtype: int64

### `.transform()` — Adding Group Statistics Back to Individual Rows

Regular GroupBy collapses your DataFrame into a summary table.
`.transform()` is different — it keeps all your original rows but **adds** the group-level statistic
as a new column next to each row.

This lets you compare each individual record against its group's benchmark —
for example: how does each employee's salary compare to their department's average?

In [171]:
# .transform()

employees['dept_avg_salary'] = employees.groupby('department')['salary'].transform('mean').round(2)

In [172]:
employees.head()

,employee_id,full_name,email,department,job_title,city,country,hire_date,salary,performance_rating,manager_id,hire_year,tenure_years,Seniority,salary_band,top_performer,dept_avg_salary
0,1001,Gregory White,gregory.white@techcorp.com,Sales,Account Manager,Mumbai,India,2021-02-06,119704.350,4.9,1006,2021,5.32,Senior,High,Yes,90780.64
1,1002,Christine Sanders,christine.sanders@techcorp.com,HR,HR Generalist,Amsterdam,Netherlands,2022-06-10,54396.610,3.6,1003,2022,3.98,Mid-Level,Low,No,93901.71
2,1003,Donald Wright,donald.wright@techcorp.com,HR,Recruiter,Cape Town,South Africa,2018-11-27,55221.150,1.9,1002,2018,7.51,Senior,Low,No,93901.71
3,1004,Richard Edwards,richard.edwards@techcorp.com,Engineering,QA Engineer,São Paulo,Brazil,2018-03-04,119468.820,2.6,1007,2018,8.25,Senior,High,No,99261.37
4,1005,Christopher Nelson,christopher.nelson@techcorp.com,Finance,Controller,Dubai,UAE,2022-06-29,105906.865,3.2,1008,2022,3.92,Mid-Level,High,No,106674.74


#### Each Employee vs Their Department Average

Now we subtract each employee's salary from their department average.
A positive number means they earn above the department average. Negative means below.

This kind of within-group comparison is only possible because `.transform()` kept all 120 rows intact.

In [174]:
employees['vs_dept_avg'] = (employees['salary'] - employees['dept_avg_salary']).round(2)

In [175]:
employees.head()

,employee_id,full_name,email,department,job_title,city,country,hire_date,salary,performance_rating,manager_id,hire_year,tenure_years,Seniority,salary_band,top_performer,dept_avg_salary,vs_dept_avg
0,1001,Gregory White,gregory.white@techcorp.com,Sales,Account Manager,Mumbai,India,2021-02-06,119704.350,4.9,1006,2021,5.32,Senior,High,Yes,90780.64,28923.71
1,1002,Christine Sanders,christine.sanders@techcorp.com,HR,HR Generalist,Amsterdam,Netherlands,2022-06-10,54396.610,3.6,1003,2022,3.98,Mid-Level,Low,No,93901.71,-39505.10
2,1003,Donald Wright,donald.wright@techcorp.com,HR,Recruiter,Cape Town,South Africa,2018-11-27,55221.150,1.9,1002,2018,7.51,Senior,Low,No,93901.71,-38680.56
3,1004,Richard Edwards,richard.edwards@techcorp.com,Engineering,QA Engineer,São Paulo,Brazil,2018-03-04,119468.820,2.6,1007,2018,8.25,Senior,High,No,99261.37,20207.45
4,1005,Christopher Nelson,christopher.nelson@techcorp.com,Finance,Controller,Dubai,UAE,2022-06-29,105906.865,3.2,1008,2022,3.92,Mid-Level,High,No,106674.74,-767.88


---

## Phase 8 — Merge, Concatenate & Datetime Operations

Every analysis so far has lived inside a single sheet. But the most valuable business questions
span **multiple sheets** at once:
- *"Which employees generated the most revenue?"* → needs `sales` + `employees`
- *"What is revenue by product category?"* → needs `sales` + `products`
- *"Show each employee's department head?"* → needs `employees` + `departments`

This phase teaches you three operations:

| Operation | What it does | Direction |
|-----------|-------------|----------|
| **Merge** | Joins two DataFrames based on a shared key column | Horizontal (adds columns) |
| **Concatenate** | Stacks DataFrames on top of each other | Vertical (adds rows) |
| **Datetime** | Deep date analysis — trends, tenure, time-between-events | On converted columns |

> ⚠️ **Important:** For this phase, we create **new variables** for all merged results.
> We do not overwrite `employees`, `dept`, `prod`, or `sales`.
> Those original DataFrames stay clean and intact.

### Understanding Merge

A merge connects two DataFrames by matching rows on a **common column** (the key).
Think of it as a VLOOKUP in Excel — but cleaner, more powerful, and works on millions of rows.

**The four merge types — `how=`:**
| Type | What it keeps |
|------|---------------|
| `'inner'` | Only rows that have a match in **both** DataFrames |
| `'left'` | All rows from the **left** DataFrame — unmatched get `NaN` in right columns |
| `'right'` | All rows from the **right** DataFrame |
| `'outer'` | All rows from **both** DataFrames |

> 🔍 **Use `how='left'` as your default.** It never drops your original rows and makes unmatched values visible as `NaN`.
> Always verify row count is the same before and after a left merge.

In [ ]:
# MERGE: join 2 dataframes ,horizontally, together based on a common key (column) - one of the most common operations in data analysis
# CONCATENATE: stacking dataframes vertically (one on top of the other)
# DATETIME: deep datetime operations using the converted columns from previous phase (5.5) 

### What We Will Merge

Here are the planned merge operations for this phase.
Each one connects two sheets via their shared key column to answer a specific business question:

- **`employees` + `dept`** → add `department_head` and `office_location` to each employee
- **`sales` + `employees`** → find which employees generated the most revenue
- **`sales` + `products`** → break down revenue by product category
- **`sales` + `employees` + `products`** → the master analysis table combining everything

The commented code below shows the target joins — implementation follows in the next cells.

In [ ]:
# employees[['full_name', 'department']] + dept['department_head']
# sales + employees